In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from loguru import logger


In [25]:
## Saving this data as .csv file to continue with Section 3: Training a machine learning algorithm
filepath = "../../data/processed/cleaned_data_final.csv"
df = pd.read_csv(filepath, encoding="utf-8")
df.head()
logger.info(f"Imported data successfully from {filepath}")

2026-03-20 21:29:16.058 | SUCCESS  | __main__:<module>:5 - Imported data successfully from ../../data/processed/cleaned_data_final.csv


In [26]:
df.drop(columns=["Equipment"], inplace=True)
df.columns.tolist()

['Brand',
 'Model',
 'Year',
 'Condition',
 'Mileage',
 'Gearbox',
 'Fiscal Power',
 'Fuel',
 'NoD',
 'Origin',
 'FO',
 'Location',
 'Sector',
 'Price',
 'num_features',
 'condition_numeric',
 'Mileage_mean']

In [27]:
# Log-transform skewed columns
df['Price_log']   = np.log1p(df['Price'])
df['Mileage_log'] = np.log1p(df['Mileage_mean'])

# Car age is more intuitive than raw year
df['Age'] = 2025 - df['Year']
# Age × Mileage interaction — an old high-mileage car is much worse
df['Age_x_Mileage'] = df['Age'] * df['Mileage_log']

# Scale numeric features
scaler = StandardScaler()
cols_to_scale = ['Year', 'Fiscal Power', 'num_features', 'Mileage_log', 'condition_numeric']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])


In [28]:
## Handeling outliers
# Train only on cars under 98th percentile price
q98 = df['Price'].quantile(0.98)
df_train_clean = df[df['Price'] <= q98]

In [29]:
# ''# 1. Define your feature matrix and target
# drop_cols = ['Price', 'Price_log', 'Condition',
#              'Mileage', 'Location', 'Sector', 'NoD']
#
# X = df_train_clean.drop(columns=drop_cols)
# y = df_train_clean['Price_log']
#
# ## Train test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )
#
#
#
# numeric_cols    = ['Year', 'Fiscal Power', 'Mileage_log', 'num_features', 'condition_numeric']
# ohe_cols        = ['Gearbox', 'Fuel', 'Origin']
# target_enc_cols = ['Brand', 'Model']
#
# preprocessor = ColumnTransformer(transformers=[
#     ('num',    StandardScaler(),                          numeric_cols),
#     ('ohe',    OneHotEncoder(drop='first',
#                handle_unknown='ignore'),                  ohe_cols),
#     ('target', TargetEncoder(target_type='continuous'),   target_enc_cols),
# ])
#
# # Then plug into your model:
#
#
# pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model',        GradientBoostingRegressor())
# ])
#
# pipeline.fit(X_train, y_train)   # y_train = Price_log
#
# # 3. Then fit your pipeline on train only
# pipeline.fit(X_train, y_train)
#
# # 4. Evaluate on test
# y_pred_log = pipeline.predict(X_test)
#
# # 5. Convert predictions back to MAD
# import numpy as np
# y_pred = np.expm1(y_pred_log)
# y_true = np.expm1(y_test)

## Using XGBoost

In [83]:
# 1. Define your feature matrix and target
drop_cols = ['Price', 'Price_log', 'Condition',
             'Mileage', 'Location', 'Sector', 'NoD']

X = df_train_clean.drop(columns=drop_cols)
y = df_train_clean['Price_log']

## Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



numeric_cols    = ['Year', 'Fiscal Power', 'Mileage_log', 'num_features', 'condition_numeric']
ohe_cols        = ['Gearbox', 'Fuel', 'Origin']
target_enc_cols = ['Brand', 'Model']



preprocessor = ColumnTransformer(transformers=[
    ('num',    'passthrough',numeric_cols),
    ('ohe',    OneHotEncoder(drop='first', handle_unknown='ignore'),ohe_cols),
    ('target', TargetEncoder(target_type='continuous'),target_enc_cols),
])

# Fit preprocessor on train, transform both
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

# model = XGBRegressor(
#     n_estimators=1000,
#     learning_rate=0.05,
#     max_depth=8,
#     reg_lambda= 5,
#     reg_alpha= 1,
#     subsample=0.9,
#     colsample_bytree=0.8,
#     early_stopping_rounds=50,
#     random_state=42,
#     device='cuda'
# )
optimized_params = {
    'subsample': 0.9,
    'reg_lambda': 5,
    'reg_alpha': 0,
    'subsample': 0.9,
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.05,
    'colsample_bytree': 0.7,
}

# Update the dict first (in-place), then unpack it
optimized_params.update({
    'device': 'cuda',
    'random_state': 42,
    'early_stopping_rounds': 50,
})

model = XGBRegressor(**optimized_params)  # unpack with **



model.fit(
    X_train_proc, y_train,
    eval_set=[(X_test_proc, y_test)],
    verbose=50
)

y_pred_log = model.predict(X_test_proc)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

[0]	validation_0-rmse:0.65978
[50]	validation_0-rmse:0.24809
[100]	validation_0-rmse:0.23302
[150]	validation_0-rmse:0.22974
[200]	validation_0-rmse:0.22809
[250]	validation_0-rmse:0.22694
[300]	validation_0-rmse:0.22596
[350]	validation_0-rmse:0.22543
[400]	validation_0-rmse:0.22500
[450]	validation_0-rmse:0.22460
[499]	validation_0-rmse:0.22428


In [84]:
from loguru import logger
# --- Metrics ---
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"MAE:  {mae:,.0f} MAD")
print(f"RMSE: {rmse:,.0f} MAD")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.2f}%")
filename = 'car_price_model.json'
model.save_model(filename)
logger.info(f"Saved model to disk filename {filename}")

MAE:  13,669 MAD
RMSE: 24,225 MAD
R²:   0.9099
MAPE: 14.74%


2026-03-20 22:54:52.054 | INFO     | __main__:<module>:14 - Saved model to disk filename car_price_model.json


In [ ]:
# Load it back later
#from xgboost import XGBRegressor
#model = XGBRegressor()
#model.load_model('car_price_model.json')

In [49]:
from sklearn.model_selection import RandomizedSearchCV

# param_grid = {
#     'n_estimators':     [500, 1000, 1500],
#     'learning_rate':    [0.01, 0.05, 0.1],
#     'max_depth':        [4, 6, 8],
#     'subsample':        [0.7, 0.8, 0.9],
#     'colsample_bytree': [0.7, 0.8, 0.9],
#     'reg_alpha':        [0, 0.1, 1],      # L1 regularization
#     'reg_lambda':       [1, 5, 10],       # L2 regularization
# }
#
#
# search = RandomizedSearchCV(
#     XGBRegressor(random_state=42, device='cuda'),
#     param_grid,
#     n_iter=50,
#     cv=5,
#     scoring='r2',
#     n_jobs=1,
#     random_state=42,
#     verbose=True
# )
#
# search.fit(X_train_proc, y_train)
# print(search.best_params_)

KeyboardInterrupt: 